<a href="https://colab.research.google.com/github/awaiskhan005/Agentic-AI/blob/main/AI_Agents_Research%2C_write_using_OpenAI_API_crewAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

You can download the `requirements.txt` for this course from the workspace of this lab. `File --> Open...`

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [ ]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [ ]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [ ]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [ ]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [ ]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [ ]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [ ]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [ ]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution.

In [ ]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [ ]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 

Content Plan: 
Title: The Latest Trends and Key Players in Artificial Intelligence

Outline:
I. Introduction
- Definition of Artificial Intelligence
- Importance of staying updated on AI trends
II. Latest Trends in Artificial Intelligence
- Machine learning advancements
- Natural language processing developments
- AI in healthcare and other industries
III. Key Players in Artificial Intelligence
- Google AI
- IBM Watson
- Amazon AI
IV. Noteworthy News in Artificial Intelligence
- Recent breakthroughs in AI te

I now can give a great answer

Final Answer:
# The Latest Trends and Key Players in Artificial Intelligence

## Introduction

Artificial Intelligence, or AI, has become a buzzword in today's technology-driven world. It refers to the simulation of human intelligence in machines that are programmed to think like humans and mimic their actions. Staying updated on AI trends is crucial for individuals and businesses alike, as it can provide valuable insights into the latest advancements and help in making informed decisions.

## Latest Trends in Artificial Intelligence

One of the most significant trends in AI is the advancements in machine learning. This technology allows machines to learn from data and improve their performance over time without being explicitly programmed. Natural language processing is another key trend, enabling machines to understand, interpret, and generate human language. Furthermore, AI is making significant strides in industries such as healthcare, where it is bei

- Display the results of your execution as markdown in the notebook.

In [ ]:
from IPython.display import Markdown
Markdown(result)

# The Latest Trends and Key Players in Artificial Intelligence

## Introduction

Artificial Intelligence, or AI, has become a buzzword in today's technology-driven world. It refers to the simulation of human intelligence in machines that are programmed to think like humans and mimic their actions. Staying updated on AI trends is crucial for individuals and businesses alike, as it can provide valuable insights into the latest advancements and help in making informed decisions.

## Latest Trends in Artificial Intelligence

One of the most significant trends in AI is the advancements in machine learning. This technology allows machines to learn from data and improve their performance over time without being explicitly programmed. Natural language processing is another key trend, enabling machines to understand, interpret, and generate human language. Furthermore, AI is making significant strides in industries such as healthcare, where it is being used for tasks like medical imaging analysis and personalized treatment recommendations.

## Key Players in Artificial Intelligence

Several companies are leading the way in AI research and development. Google AI is at the forefront of AI innovation, with projects like AlphaGo and Duplex showcasing the capabilities of AI technology. IBM Watson is another key player, known for its cognitive computing capabilities and applications in various industries. Amazon AI, with its Alexa virtual assistant, is also making waves in the AI space.

## Noteworthy News in Artificial Intelligence

Recent breakthroughs in AI technology include advancements in autonomous vehicles, robotics, and computer vision. These developments have the potential to revolutionize various industries and improve efficiency and productivity. However, the impact of AI on society is a topic of ongoing debate, with concerns about job displacement, privacy issues, and ethical considerations.

## Target Audience Analysis

Our target audience includes tech enthusiasts interested in AI, business professionals looking to implement AI solutions, and students studying AI or related fields. By providing insights into the latest trends and key players in AI, we aim to help our audience stay informed and ahead in this rapidly evolving field.

## SEO Keywords: 
- Artificial Intelligence trends
- Key players in AI
- Latest AI news

In conclusion, Artificial Intelligence is transforming the way we live and work, with new trends and key players shaping the future of technology. By staying informed and engaging with the latest developments in AI, individuals and businesses can harness the power of this groundbreaking technology for innovation and growth. Stay updated on the latest trends and key players in Artificial Intelligence by subscribing to our newsletter for regular updates and insights.

References:
- Research papers on AI trends
- Reports from industry experts
- Case studies on successful AI implementations

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [ ]:
topic = "Freelancing"
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Freelancing.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer:
Content Plan on Freelancing

1. Latest Trends, Key Players, and Noteworthy News on Freelancing
- Latest Trends: The rise of remote work, gig economy growth, and the increasing demand for freelancers in various industries.
- Key Players: Popular freelancing platforms such as Upwork, Fiverr, and Freelancer.com, as well as notable freelancers who have made a name for themselves in their respective fields.
- Noteworthy News: Recent studies on the impact of freelancing on the economy, success stories of freelancers, and new to

I now can give a great answer

Final Answer:
# The Modern Workforce: Navigating the Freelancing Landscape

## Introduction

In today's rapidly evolving work environment, freelancing has emerged as a popular and viable option for many individuals seeking flexibility and autonomy in their careers. Defined as the practice of working independently on a project-to-project basis, freelancing offers unique opportunities for professionals to showcase their skills, pursue their passions, and create a work-life balance that suits their needs. This article explores the latest trends and key players in the freelancing industry, shedding light on the dynamic landscape of remote work and the gig economy.

## How to Start Freelancing

For aspiring freelancers looking to venture into the world of self-employment, the journey can be both exciting and daunting. To kickstart a successful freelancing career, it is essential to identify your niche, build a strong portfolio that showcases your expertise, an

In [ ]:
Markdown(result)

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).